# A3. Recommendation System

MLE: Estimación de Máxima Verosimilitud

Pedro Mauri Mtz - A01029143

In [37]:
import numpy as np
from scipy.special import softmax

np.random.seed(0)

In [38]:
"""
samples = np.random.choice([0, 1, 2], size = 10000, p = [0.6, 0.3, 0.1])

unique_values, counts = np.unique(samples, return_counts = True)

counts/counts.sum()

params = np.zeros(3)
"""

'\nsamples = np.random.choice([0, 1, 2], size = 10000, p = [0.6, 0.3, 0.1])\n\nunique_values, counts = np.unique(samples, return_counts = True)\n\ncounts/counts.sum()\n\nparams = np.zeros(3)\n'

Categorias, subcategorias y peliculas singulares definidas

In [39]:
categories = ["Acción", "Aventura", "Romance"]

subcategories = {
    "Acción": ["Superhéroes", "Policial", "Artes Marciales"],
    "Aventura": ["Fantasía", "Sci-Fi", "Histórica"],
    "Romance": ["Comedia Rom.", "Drama", "Épica"]
}

items = {
    ("Acción", "Superhéroes"): ["Avengers", "The Dark Knight", "Iron Man"],
    ("Acción", "Policial"): ["John Wick", "Heat", "The Raid 2"],
    ("Acción", "Artes Marciales"): ["Ip Man", "Hero", "The Raid"],
    ("Aventura", "Fantasía"): ["LOTR", "Willow", "The Hobbit"],
    ("Aventura", "Sci-Fi"): ["Dune", "Interstellar", "The Martian"],
    ("Aventura", "Histórica"): ["Gladiator", "Braveheart", "Troy"],
    ("Romance", "Comedia Rom."): ["Notting Hill", "10 Things I Hate About You", "Love Actually"],
    ("Romance", "Drama"): ["La La Land", "A Star is Born", "Brooklyn"],
    ("Romance", "Épica"): ["Titanic", "Pride & Prejudice", "Atonement"]
}

Parámetros iniciales

In [40]:
# Distribución por categorías
params_cat = np.zeros(len(categories))
# Distribución por subcategoría
params_sub = {cat: np.zeros(len(subcategories[cat])) for cat in categories}

# Preferencias "ocultas" del usuario (para simular recompensas)
true_pref_cat = np.array([0.6, 0.3, 0.1])
true_pref_sub = {
    "Acción": np.array([0.7, 0.2, 0.1]),
    "Aventura": np.array([0.3, 0.4, 0.3]),
    "Romance": np.array([0.2, 0.6, 0.2])
}

In [41]:
def sample_category(params):

    probs = softmax(params)

    return np.random.choice(categories, p = probs), probs

In [42]:
def sample_subcategory(params_sub_cat, cat):

    probs = softmax(params_sub_cat)

    return np.random.choice(subcategories[cat], p = probs), probs

In [43]:
def user_reward(cat, sub):

    p_cat = true_pref_cat[categories.index(cat)]
    p_sub = true_pref_sub[cat][subcategories[cat].index(sub)]
    p_like = 0.5 * p_cat + 0.5 * p_sub

    return 1.0 if np.random.rand() < p_like else 0.0

Entrenamiento Reinforce

In [44]:
lr = 0.001
baseline = 0.0
rewards_hist = []
T = 10000

for t in range(T):

    # 1. Elegir categoría
    cat, p_cat = sample_category(params_cat)

    # 2. Elegir subcategoría dentro de la categoría
    sub, p_sub = sample_subcategory(params_sub[cat], cat)

    # 3. Simular recompensa
    reward = user_reward(cat, sub)
    rewards_hist.append(reward)

    # Actualiza el baseline como promedio móvil de la recompensa
    baseline = 0.99 * baseline + 0.01 * reward
    advantage = reward - baseline  # ahora puede ser negativo

    # 4. Gradiente REINFORCE para categoría
    grad_cat = np.zeros_like(params_cat)
    grad_cat[categories.index(cat)] = 1
    grad_cat -= p_cat
    params_cat += lr * grad_cat * advantage

    # 5. Gradiente REINFORCE para subcategoría
    grad_sub = np.zeros_like(params_sub[cat])
    grad_sub[subcategories[cat].index(sub)] = 1
    grad_sub -= p_sub
    params_sub[cat] += lr * grad_sub * advantage

"""
for i in range(T):
    probs = softmax(params)
    sample = np.random.choice([0, 1, 2], p = probs)

    # Recompensa: sólo 2 da premio
    reward = 1.0 if sample == 2 else 0.0

    # Gradiente REINFORCE: (one-hot - probs)
    grad = np.zeros_like(params)
    grad[sample] = 1
    grad -= probs

    # Actualización
    params += lr * grad * reward
    rewards_hist.append(reward)

print("Parámetros finales:", params)
print("Distribución aprendida:", softmax(params))
print("Recompensa promedio:", np.mean(rewards_hist))
"""


'\nfor i in range(T):\n    probs = softmax(params)\n    sample = np.random.choice([0, 1, 2], p = probs)\n\n    # Recompensa: sólo 2 da premio\n    reward = 1.0 if sample == 2 else 0.0\n\n    # Gradiente REINFORCE: (one-hot - probs)\n    grad = np.zeros_like(params)\n    grad[sample] = 1\n    grad -= probs\n\n    # Actualización\n    params += lr * grad * reward\n    rewards_hist.append(reward)\n\nprint("Parámetros finales:", params)\nprint("Distribución aprendida:", softmax(params))\nprint("Recompensa promedio:", np.mean(rewards_hist))\n'

Prints de resultados

In [45]:
print("=== Distribución aprendida ===")

for cat in categories:
    print(f"\nCategoría {cat}: p ≈ {softmax(params_cat)[categories.index(cat)]:.3f}")

    for sub, p in zip(subcategories[cat], softmax(params_sub[cat])):
        print(f"   - {sub:15s}: {p:.3f}")

print("\nRecompensa promedio:", np.mean(rewards_hist))

=== Distribución aprendida ===

Categoría Acción: p ≈ 0.514
   - Superhéroes    : 0.433
   - Policial       : 0.296
   - Artes Marciales: 0.271

Categoría Aventura: p ≈ 0.271
   - Fantasía       : 0.336
   - Sci-Fi         : 0.338
   - Histórica      : 0.326

Categoría Romance: p ≈ 0.215
   - Comedia Rom.   : 0.312
   - Drama          : 0.375
   - Épica          : 0.312

Recompensa promedio: 0.3645
